# 02 · Whisper transcription & JSONL

**Portfolio adaptation of the 2023 SJK Whisper transcription notebook.** Maps downloaded MP3 filenames to video metadata, transcribes audio using Whisper, and writes overlapping timestamped JSONL chunks for downstream indexing.

The original sanitized notebook is under `notebooks/sanitized_reference/audio_tran.ipynb`. This featured notebook removes unrelated Hugging Face tutorial cells, fixes chunk boundaries, and **defaults to a deterministic fake transcription** so it can be validated without downloading a Whisper model or using private media. Set `SJK_DEMO_MODE=0` to use a locally installed Whisper model.

In [ ]:
from pathlib import Path
import csv
import json
import os

PROJECT_DIR = Path(os.environ.get("SJK_PORTFOLIO_DIR", "/content/sjk_portfolio"))
AUDIO_DIR = PROJECT_DIR / "audio" / "mp3_converted"
METADATA_CSV = PROJECT_DIR / "audio" / "metadata.csv"
OUTPUT_JSONL = PROJECT_DIR / "transcription.jsonl"
DEMO_MODE = os.environ.get("SJK_DEMO_MODE", "1") == "1"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print("Source MP3 directory:", AUDIO_DIR, "Demo mode:", DEMO_MODE)


## Build the metadata lookup and timestamped transcript windows
The 2023 notebook associated each MP3's video ID with CSV metadata, then split Whisper segments into overlapping windows.


In [ ]:
def load_video_metadata(metadata_csv):
    if not Path(metadata_csv).exists():
        raise FileNotFoundError(f"Missing metadata CSV: {metadata_csv}")
    with open(metadata_csv, newline="", encoding="utf-8") as f:
        return {row["Video ID"]: row for row in csv.DictReader(f)}


def window_segments(segments, video_id, metadata=None, window=10, stride=3):
    if window <= 0 or stride <= 0:
        raise ValueError("window and stride must be positive")
    metadata = metadata or {}
    for start_index in range(0, len(segments), stride):
        segment_slice = segments[start_index:start_index + window]
        if not segment_slice:
            continue
        start = segment_slice[0]["start"]
        end = segment_slice[-1]["end"]
        yield {**metadata, "id": f"{video_id}-t{start}",
               "text": " ".join(part["text"].strip() for part in segment_slice).strip(),
               "start": start, "end": end}


def transcribe_to_jsonl(model, audio_dir=AUDIO_DIR, metadata_csv=METADATA_CSV,
                       output_jsonl=OUTPUT_JSONL, window=10, stride=3):
    """Process MP3 files; model.transcribe() is interchangeable with a test double."""
    info = load_video_metadata(metadata_csv)
    audio_files = sorted(Path(audio_dir).glob("*.mp3"))
    Path(output_jsonl).parent.mkdir(parents=True, exist_ok=True)
    count = 0
    with open(output_jsonl, "w", encoding="utf-8") as f:
        for audio_file in audio_files:
            video_id = audio_file.stem
            result = model.transcribe(str(audio_file))
            for chunk in window_segments(result.get("segments", []), video_id,
                                         info.get(video_id, {}), window, stride):
                f.write(json.dumps(chunk, ensure_ascii=False) + "\n")
                count += 1
    return count


## Test with synthetic transcript segments (default) or real Whisper (opt-in)
Whisper is intentionally not installed or invoked in demo mode. To run a genuine transcription in Colab: install `openai-whisper` and `ffmpeg`, then set `SJK_DEMO_MODE=0` and optionally `SJK_WHISPER_MODEL=base` (the historical notebook used `large`).


In [ ]:
if DEMO_MODE:
    class SyntheticTranscriber:
        def transcribe(self, path):
            return {"segments": [
                {"start": 0.0, "end": 1.0, "text": "Synthetic sample."},
                {"start": 1.0, "end": 2.0, "text": "No speech recognition was run."},
                {"start": 2.0, "end": 3.0, "text": "For repeatable tests."},
            ]}
    model = SyntheticTranscriber()
else:
    import whisper
    model_name = os.environ.get("SJK_WHISPER_MODEL", "base")
    model = whisper.load_model(model_name)

if AUDIO_DIR.exists() and METADATA_CSV.exists():
    records_written = transcribe_to_jsonl(model, window=2, stride=1)
    print("Transcript chunks written:", records_written, "to", OUTPUT_JSONL)
else:
    print("Run the video extraction demo first, or supply your own reviewed MP3 and metadata CSV.")


## Limits
The synthetic transcriber checks JSONL organization, joining metadata to filenames, and chunk/timestamp logic. **It does not validate speech-to-text accuracy.** A full Whisper model run remains a separate manual Colab verification step.
